# 3a. Contract negotiation (DSP 2025-1, eunomia RPC)

Before any data is transferred, the Consumer and the Provider must reach a contract agreement on a
dataset. This notebook walks the full Dataspace Protocol 2025-1 negotiation state machine
(DSP 8.1) through the eunomia RPCs (`/dsp/current/negotiations/rpc/*`) and then inspects the
persisted processes, messages, offers and agreement through the Negotiation Agent Management API
(`/api/v1/negotiation-agent/*`):

| Step | Sender | RPC (`/dsp/current/negotiations/rpc/...`) | DSP message | State |
|---|---|---|---|---|
| 2 | Consumer | `setup-request-init` | `ContractRequestMessage` (DSP 8.2.2) | `REQUESTED` |
| 3 | Provider | `setup-offer` | `ContractOfferMessage` (DSP 8.3.4) | `OFFERED` |
| 4 | Consumer | `setup-request` | `ContractRequestMessage` (DSP 8.2.3) | `REQUESTED` |
| 5 | Provider | `setup-offer` | `ContractOfferMessage` (DSP 8.3.4) | `OFFERED` |
| 6 | Consumer | `setup-acceptance` | `ContractNegotiationEventMessage` `ACCEPTED` (DSP 8.2.4) | `ACCEPTED` |
| 7 | Provider | `setup-agreement` | `ContractAgreementMessage` (DSP 8.3.5) | `AGREED` |
| 8 | Consumer | `setup-verification` | `ContractAgreementVerificationMessage` (DSP 8.2.5) | `VERIFIED` |
| 9 | Provider | `setup-finalization` | `ContractNegotiationEventMessage` `FINALIZED` (DSP 8.3.6) | `FINALIZED` |

**Runs on its own.** It leaves `agreement_id` for the transfer notebooks, and records
`agreement_source = ("rpc", target_id)` so that a transfer notebook can tell whether the agreement
in the kernel belongs to the loaded dataset.

## 1. Prerequisites

The guard populates a catalog only when no catalog notebook has run in this kernel; it runs
`setup_catalog_policies.ipynb`, which in turn onboards the participants if needed. Run
`setup_catalog_policies_oauth.ipynb` first to negotiate on its dataset instead.

In [ ]:
if not globals().get("catalog_setup"):
    %run ./setup_catalog_policies.ipynb

### How the negotiation RPCs and Management API work

Neither participant calls the peer's DSP endpoints directly. Each one asks **its own agent** to
send the next DSP message through an RPC under `/dsp/current/negotiations/rpc/`. The body is
eunomia JSON-LD:

```json
{
  "@context": "https://w3id.org/eunomia/v1/context.jsonld",
  "@type": "RpcNegotiation<Kind>Message",
  "consumerPid": "...", "providerPid": "...",
  "offer": { ... }
}
```

The agent reads it through RDF, builds the DSP message, checks it against the negotiation state
machine and its own role, sends it with its peer token and stores the result once the peer
accepts it. Every RPC answers an `RpcNegotiationResult`:
- `request`: the RPC body as received.
- `response`: the peer's answer (its `ContractNegotiation` ACK for an opening message).
- `negotiationAgentModel`: the caller's own `NegotiationProcessView`, with `id`, `state`, `role`,
  `protocol`, and `correlation` (`consumerPid`, `providerPid`, `callbackAddress`,
  `peerParticipantId`, `identifiers`).

The associated records of a negotiation are queried through the Management API under
`/api/v1/negotiation-agent/`:
- `/negotiation-processes/{id}`: the negotiation process record and correlation.
- `/negotiation-messages/process/{id}`: every DSP message sent or received, with its state transition.
- `/offers/process/{id}`: every ODRL offer proposed during the negotiation.
- `/agreements/process/{id}` and `/agreements/{agreement_id}`: the finalized contract agreement.

An opening RPC answers 201, the rest 200. Errors answer a `ContractNegotiationError`: 400 for a
broken rule, 404 for an unknown negotiation, 409 for a stale or repeated message, 502 when the
peer refused it.

In [ ]:
def negotiation_step(response, expected_state):
    """Prints an RpcNegotiationResult and checks the caller's process reached expected_state."""
    answer = show(response)
    process = answer["negotiationAgentModel"]
    assert process["state"] == expected_state, f"expected {expected_state}, got {process['state']}"
    print(f"\n==> {process['state']}")
    return process

## 2. Contract request (Consumer -> Provider)

The Consumer opens the negotiation on the published offer: `@id` is the offer from the catalog
and `target` the dataset that publishes it. The Provider rejects a request on an offer it did not
publish, or on another dataset. `providerAddress` is where the request goes and `callbackAddress`
where the Provider answers. Both PIDs are returned in `correlation`.

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcNegotiationRequestInitMessage",
    "associatedAgentPeer": data_space_provider_participant_id,
    "providerAddress": data_space_provider_dsp_endpoint,
    "callbackAddress": data_space_consumer_dsp_endpoint,
    "offer": {
        "@id": policy_id,
        "@type": "Offer",
        "target": target_id,
        "permission": [{"action": "use"}],
    },
}
response = requests.post(f"{data_space_consumer}/dsp/current/negotiations/rpc/setup-request-init", headers=json_header, json=payload)
consumer_process = negotiation_step(response, "REQUESTED")
cn_consumer_id = consumer_process["correlation"]["consumerPid"]
cn_provider_id = consumer_process["correlation"]["providerPid"]

def own_negotiation_id(agent_url, consumer_pid):
    """The id an agent gives the negotiation whose consumerPid is `consumer_pid`."""
    processes = requests.get(f"{agent_url}/api/v1/negotiation-agent/negotiation-processes", headers=json_header).json()
    return next(p["id"] for p in processes["items"] if p["correlation"]["consumerPid"] == consumer_pid)

# RPC calls name the negotiation by each agent's own process id, never by DSP pids.
cn_consumer_process_id = consumer_process["id"]
cn_provider_process_id = own_negotiation_id(data_space_provider, cn_consumer_id)
print(f"consumer process: {cn_consumer_process_id}\nprovider process: {cn_provider_process_id}")

## 3. Counter-offer (Provider -> Consumer)

The Provider answers with different terms: use is limited to 1000 requests.

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcNegotiationOfferMessage",
    "@id": cn_provider_process_id,
    "offer": {
        "@id": policy_id,
        "@type": "Offer",
        "target": target_id,
        "permission": [{
            "action": "use",
            "constraint": [{"leftOperand": "count", "operator": "lteq", "rightOperand": "1000"}],
        }],
    },
}
response = requests.post(f"{data_space_provider}/dsp/current/negotiations/rpc/setup-offer", headers=json_header, json=payload)
provider_process = negotiation_step(response, "OFFERED")

## 4. Counter-request (Consumer -> Provider)

The Consumer does not accept the limit and asks again for unrestricted use.

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcNegotiationRequestMessage",
    "@id": cn_consumer_process_id,
    "offer": {
        "@id": policy_id,
        "@type": "Offer",
        "target": target_id,
        "permission": [{"action": "use"}],
    },
}
response = requests.post(f"{data_space_consumer}/dsp/current/negotiations/rpc/setup-request", headers=json_header, json=payload)
consumer_process = negotiation_step(response, "REQUESTED")

## 5. Final offer (Provider -> Consumer)

The Provider gives in and offers the terms the Consumer asked for. This is the offer the agreement
will carry.

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcNegotiationOfferMessage",
    "@id": cn_provider_process_id,
    "offer": {
        "@id": policy_id,
        "@type": "Offer",
        "target": target_id,
        "permission": [{"action": "use"}],
    },
}
response = requests.post(f"{data_space_provider}/dsp/current/negotiations/rpc/setup-offer", headers=json_header, json=payload)
provider_process = negotiation_step(response, "OFFERED")

## 6. Acceptance (Consumer -> Provider)

The Consumer accepts the last offer. The body only names the negotiation: `@type` says which
message goes out.

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcNegotiationAcceptanceMessage",
    "@id": cn_consumer_process_id,
}
response = requests.post(f"{data_space_consumer}/dsp/current/negotiations/rpc/setup-acceptance", headers=json_header, json=payload)
consumer_process = negotiation_step(response, "ACCEPTED")

## 7. Agreement (Provider -> Consumer)

The Provider turns the accepted offer into a `ContractAgreementMessage` and sends it.

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcNegotiationAgreementMessage",
    "@id": cn_provider_process_id,
}
response = requests.post(f"{data_space_provider}/dsp/current/negotiations/rpc/setup-agreement", headers=json_header, json=payload)
provider_process = negotiation_step(response, "AGREED")

## 8. Verification (Consumer -> Provider)

The Consumer confirms it received the agreement and accepts it as binding.

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcNegotiationVerificationMessage",
    "@id": cn_consumer_process_id,
}
response = requests.post(f"{data_space_consumer}/dsp/current/negotiations/rpc/setup-verification", headers=json_header, json=payload)
consumer_process = negotiation_step(response, "VERIFIED")

## 9. Finalization (Provider -> Consumer)

The Provider closes the negotiation. `FINALIZED` is terminal: the agreement can now be used to
request transfers.

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcNegotiationFinalizationMessage",
    "@id": cn_provider_process_id,
}
response = requests.post(f"{data_space_provider}/dsp/current/negotiations/rpc/setup-finalization", headers=json_header, json=payload)
provider_process = negotiation_step(response, "FINALIZED")

## 10. Inspecting processes, messages, offers and agreement (`/api/v1/negotiation-agent/*`)

The Management API of each agent exposes its persisted records under `/api/v1/negotiation-agent/`:
- `GET /negotiation-processes/{id}`: the process state and `correlation` (`consumerPid`, `providerPid`, `peerParticipantId`).
- `GET /negotiation-messages/process/{id}`: the DSP messages exchanged in chronological order.
- `GET /offers/process/{id}`: the offers exchanged during the negotiation.
- `GET /agreements/process/{id}` and `GET /agreements/{agreement_id}`: the resulting agreement, now
  with `assigner` (Provider), `assignee` (Consumer) and state `ACTIVE`. Both agents keep the same
  `agreement_id`, which is the `agreementId` a transfer request names.

In [ ]:
provider_agreements = show(requests.get(
    f"{data_space_provider}/api/v1/negotiation-agent/agreements/process/{provider_process['id']}",
    headers=json_header,
))
agreement = provider_agreements["items"][0]
agreement_id = agreement["id"]

for name, agent_url, process in [("Consumer", data_space_consumer, consumer_process), ("Provider", data_space_provider, provider_process)]:
    proc_view = requests.get(f"{agent_url}/api/v1/negotiation-agent/negotiation-processes/{process['id']}", headers=json_header).json()
    msgs_view = requests.get(f"{agent_url}/api/v1/negotiation-agent/negotiation-messages/process/{process['id']}", headers=json_header).json()
    offers_view = requests.get(f"{agent_url}/api/v1/negotiation-agent/offers/process/{process['id']}", headers=json_header).json()
    agr_view = requests.get(f"{agent_url}/api/v1/negotiation-agent/agreements/{agreement_id}", headers=json_header).json()
    transitions = [f"{m.get('stateTransitionFrom') or 'Initial'}->{m['stateTransitionTo']}" for m in msgs_view["items"]]
    print(f"{name}: state={proc_view['state']}, offers={len(offers_view['items'])}, messages={len(msgs_view['items'])} {transitions}, agreement={agr_view['id']} ({agr_view['state']})")

In [ ]:
# Marker read by the guards of the transfer notebooks
agreement_source = ("rpc", target_id)
print(f"agreement_id = {agreement_id}")

## Optional: termination

Either side may terminate a negotiation that is not finished yet with `setup-termination`
(`RpcNegotiationTerminationMessage`, with an optional `code` and `reason`), which moves it to
`TERMINATED` (DSP 8.2.6, 8.3.7). This one is `FINALIZED`, so the agent refuses: the cell shows the
`ContractNegotiationError` it answers.

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcNegotiationTerminationMessage",
    "@id": cn_consumer_process_id,
    "code": "TUTORIAL",
    "reason": ["The negotiation is already finalized"],
}
answer = show(requests.post(f"{data_space_consumer}/dsp/current/negotiations/rpc/setup-termination", headers=json_header, json=payload), ok=False)
assert answer["@type"] == "ContractNegotiationError"